In [ ]:
import sys
import time
from multiprocessing import Process, Event, Value, Queue
from datetime import datetime
from dataclasses import dataclass
import csv
import math
import typing
import numpy
import queue
from itertools import pairwise

import serial
from scipy.interpolate import (
    NearestNDInterpolator,
    LinearNDInterpolator,
    CloughTocher2DInterpolator,
)
import pandas
from sensirion_shdlc_driver import ShdlcSerialPort, ShdlcConnection
from sensirion_uart_scc1.drivers.scc1_slf3x import Scc1Slf3x
from sensirion_uart_scc1.drivers.slf_common import get_flow_unit_label
from sensirion_uart_scc1.scc1_shdlc_device import Scc1ShdlcDevice
from simple_pid import PID


def async_print(
    print_queue: Queue,
    message: str,
    is_erasable: bool = False,
    should_erase_previous_now=False,
    block: bool = False,
    timeout: typing.Optional[float] = None,
) -> bool:
    try:
        print_queue.put(
            (message, is_erasable, should_erase_previous_now),
            block=block,
            timeout=timeout,
        )
        return True
    except queue.Full:
        return False


def delete_printed_lines(number_of_lines: int):
    for _ in range(number_of_lines):
        # \033[F moves the cursor to the beginning of the previous line
        # \033[K clears the line from the cursor to the end
        sys.stdout.write("\033[F\033[K")


class Timer:
    def __init__(
        self,
        offset_to_add: float = 0.0,
    ) -> None:
        self.start(offset_to_add=offset_to_add)

    def start(self, offset_to_add: float = 0.0) -> None:
        self.start_time = time.time() + offset_to_add

    def time(self) -> float:
        return time.time() - self.start_time


@dataclass
class CommandExecutionData:
    start_time: float
    end_time: float
    gcode: str


class Pump:
    def __init__(
        self,
        serial_port_label: str,
        print_queue,
        serial_port_baudrate: int = 115200,
        x_offset_to_add: float = 0.00,
        z_offset_to_add: float = 3.00,
    ) -> None:
        self.x_offset_to_add: float = x_offset_to_add
        self.z_offset_to_add: float = z_offset_to_add
        self.print_queue = print_queue

        async_print(
            print_queue=self.print_queue, message="Initializing the GRBL serial port..."
        )
        self.serial_port: serial.Serial = serial.Serial(
            port=serial_port_label,
            baudrate=serial_port_baudrate,
        )
        async_print(
            print_queue=self.print_queue,
            message="Waiting for GRBL serial port to open...",
        )
        time.sleep(3)
        async_print(print_queue=self.print_queue, message="GRBL serial port ready...")

        self.timer = Timer()

    def wait_for_acknowledgement(self) -> str:
        response: str = ""

        while "ok" not in response.lower() and "error" not in response.lower():
            time.sleep(0.005)
            try:
                response = self.serial_port.readline().decode().strip()
            except Exception as exception:
                async_print(
                    print_queue=self.print_queue,
                    message=f"Error while waiting for GRBL acknowledgement: {exception}",
                )
        # async_print(print_queue=self.print_queue, message= "GRBL Acknowledgement received...")

        self.serial_port.reset_input_buffer()

        return response

    def wait_for_idle(self) -> str:
        response: str = ""

        while "idle" not in response.lower():
            time.sleep(0.005)
            try:
                self.serial_port.reset_input_buffer()
                self.serial_port.write("?\n".encode())
                response = self.serial_port.readline().decode().strip()
            except Exception as exception:
                async_print(
                    print_queue=self.print_queue,
                    message=f"Error while waiting for GRBL Controller idle status: {exception}",
                )
        # async_print(print_queue=self.print_queue, message= "GRBL Control System idle status received...")

        self.serial_port.reset_input_buffer()

        return response

    def execute_gcode(
        self,
        gcode: str,
        should_wait_for_acknowledgement: bool = True,
        should_wait_for_idle: bool = True,
        should_log: bool = True,
    ) -> list[CommandExecutionData]:
        if should_log:
            async_print(print_queue=self.print_queue, message=f"GCode> {gcode}")

        start_time = self.timer.time()
        self.serial_port.write(f"{gcode}\n".encode())
        if should_wait_for_acknowledgement:
            self.wait_for_acknowledgement()
        if should_wait_for_idle:
            self.wait_for_idle()
        end_time = self.timer.time()

        return [
            CommandExecutionData(
                start_time=start_time,
                end_time=end_time,
                gcode=gcode,
            )
        ]

    def pause(
        self,
        duration: float,
        should_wait_for_acknowledgement: bool = True,
        should_log: bool = True,
    ) -> list[CommandExecutionData]:
        return self.execute_gcode(
            gcode=f"G04 P{duration:0.2f}",
            should_wait_for_acknowledgement=should_wait_for_acknowledgement,
            should_wait_for_idle=False,
            should_log=should_log,
        )

    def move_x(
        self,
        position: float,
        feedrate: float,
        should_wait_for_acknowledgement: bool = True,
        should_log: bool = True,
    ) -> list[CommandExecutionData]:
        return self.execute_gcode(
            gcode=f"G01 X{position + self.x_offset_to_add:0.5f} F{feedrate:0.3f}",
            should_wait_for_acknowledgement=should_wait_for_acknowledgement,
            should_wait_for_idle=True,
            should_log=should_log,
        )

    def move_z(
        self,
        position: float,
        feedrate: float,
        should_wait_for_acknowledgement: bool = True,
        should_log: bool = True,
    ) -> list[CommandExecutionData]:
        return self.execute_gcode(
            gcode=f"G01 Z{position + self.z_offset_to_add:0.5f} F{feedrate:0.3f}",
            should_wait_for_acknowledgement=should_wait_for_acknowledgement,
            should_wait_for_idle=True,
            should_log=should_log,
        )

    def reset(self, should_log: bool = False) -> list[CommandExecutionData]:
        command_execution_data_list: list[CommandExecutionData] = []
        if should_log:
            async_print(print_queue=self.print_queue, message="Homing the Pump...")

        command_execution_data_list += self.execute_gcode(
            gcode="$H", should_log=should_log
        )
        command_execution_data_list += self.execute_gcode(
            gcode="G92 X0.0 Y0.0 Z0.0", should_log=should_log
        )

        command_execution_data_list += self.move_x(
            position=3.8, feedrate=400.0, should_log=should_log
        )

        command_execution_data_list += self.move_z(
            position=0.0, feedrate=1000.0, should_log=should_log
        )

        if should_log:
            async_print(print_queue=self.print_queue, message="Ready...")

        return command_execution_data_list

    def execute_triangle(
        self,
        plunger_position_apex,
        plunger_position_trough,
        feedrate,
        should_log: bool = True,
    ) -> list[CommandExecutionData]:
        command_execution_data_list: list[CommandExecutionData] = []

        command_execution_data_list += self.move_z(
            position=plunger_position_apex,
            feedrate=feedrate,
            should_log=should_log,
        )
        command_execution_data_list += self.move_z(
            position=plunger_position_trough,
            feedrate=feedrate,
            should_log=should_log,
        )

        return command_execution_data_list

    def close(self, should_log: bool = False) -> list[CommandExecutionData]:
        command_execution_data_list: list[CommandExecutionData] = []

        command_execution_data_list += self.move_z(
            position=0.0, feedrate=400.0, should_log=should_log
        )
        command_execution_data_list += self.move_x(
            position=3.8, feedrate=400.0, should_log=should_log
        )

        if (self.serial_port is not None) and (self.serial_port.is_open):
            async_print(print_queue=self.print_queue, message="")
            async_print(
                print_queue=self.print_queue, message="\nClosing Pump Serial Port..."
            )

            self.serial_port.close()

        return command_execution_data_list


@dataclass
class PressureSensorData:
    time: float
    value: float


class PressureSensor:
    def __init__(
        self,
        serial_port_label: str,
        print_queue,
        serial_port_baudrate: int = 9600,
        offset_add_to_reading_value: float = 0.0,
    ) -> None:
        self.offset_add_to_reading_value: float = offset_add_to_reading_value
        self.print_queue = print_queue

        async_print(
            print_queue=self.print_queue,
            message="Initializing the Pressure Sensor serial port...",
        )
        self.serial_port: serial.Serial = serial.Serial(
            port=serial_port_label,
            baudrate=serial_port_baudrate,
        )
        async_print(
            print_queue=self.print_queue,
            message="Waiting for Pressure Sensor serial port to open...",
        )
        time.sleep(3)
        async_print(
            print_queue=self.print_queue, message="Pressure Sensor serial port ready..."
        )

        self.timer = Timer()

    def read(self, should_log: bool = False) -> typing.Optional[PressureSensorData]:
        try:
            reading_value = (
                float(self.serial_port.readline().decode().strip().split(",")[1])
                + self.offset_add_to_reading_value
            )
            reading_time = self.timer.time()

            if should_log:
                async_print(
                    print_queue=self.print_queue,
                    message=f"Pressure Sensor Reading: (Time: {reading_time}, Value: {reading_value})",
                )

            return PressureSensorData(
                time=reading_time,
                value=reading_value,
            )
        except Exception as exception:
            async_print(
                print_queue=self.print_queue,
                message=f"Error reading pressure sensor: {exception}",
            )
            return None

    def read_averaged(
        self, number_of_samples: int = 4, should_log: bool = False
    ) -> typing.Optional[PressureSensorData]:
        times: list[float] = []
        values: list[float] = []
        max_attempts: int = number_of_samples * 2

        for _ in range(max_attempts):
            if len(values) >= number_of_samples:
                break

            sample: typing.Optional[PressureSensorData] = self.read(
                should_log=should_log
            )
            if sample is not None:
                times.append(sample.time)
                values.append(sample.value)

        if len(values) == 0:
            return None

        return PressureSensorData(
            time=sum(times) / len(times),
            value=sum(values) / len(values),
        )

    def close(self) -> None:
        if (self.serial_port is not None) and (self.serial_port.is_open):
            async_print(print_queue=self.print_queue, message="")
            async_print(
                print_queue=self.print_queue,
                message="Closing Pressure Sensor Serial Port...",
            )

            self.serial_port.close()


@dataclass
class FlowRateSensorData:
    time: float
    value: float


class FlowRateSensor:
    def __init__(
        self,
        serial_port_label: str,
        print_queue,
        serial_port_baudrate: int = 115200,
        continuous_measurement_interval: int = 5,
        offset_add_to_reading_value: float = 0.0,
    ) -> None:
        self.offset_add_to_reading_value: float = offset_add_to_reading_value
        self.print_queue = print_queue

        try:
            async_print(
                print_queue=self.print_queue,
                message="Initializing the Flow Rate Sensor serial port...",
            )
            self.serial_port: ShdlcSerialPort = ShdlcSerialPort(
                port=serial_port_label,
                baudrate=serial_port_baudrate,
                do_open=True,
            )
            device = Scc1ShdlcDevice(ShdlcConnection(self.serial_port), slave_address=0)
            device.set_sensor_type(Scc1Slf3x.SENSOR_TYPE)
            self.sensor = Scc1Slf3x(device)
            flow_unit_and_scale = self.sensor.get_flow_unit_and_scale()
            if flow_unit_and_scale is not None:
                self.flow_rate_scale, unit = flow_unit_and_scale
            else:
                self.flow_rate_scale = 1.0
            self.sensor.start_continuous_measurement(
                interval_ms=continuous_measurement_interval
            )
            async_print(
                print_queue=self.print_queue,
                message="Waiting for Flow Rate Sensor serial port to open...",
            )
            time.sleep(3)
            async_print(
                print_queue=self.print_queue,
                message="Flow Rate Sensor serial port ready...",
            )

            self.timer = Timer()
        except Exception as exception:
            async_print(
                print_queue=self.print_queue,
                message=f"Error while initializing the Flow Sensor: {exception}",
            )

    def read(self, should_log: bool = False) -> typing.Optional[FlowRateSensorData]:
        try:
            reading_value = None
            remaining, lost, data = self.sensor.read_extended_buffer()
            for flow_rate, _, _ in data:
                reading_value = (
                    flow_rate / self.flow_rate_scale + self.offset_add_to_reading_value
                )
            reading_time = self.timer.time()
            if reading_value is None:
                return None

            if should_log:
                async_print(
                    print_queue=self.print_queue,
                    message=f"Flow Sensor Reading: (Time: {reading_time}, Value: {reading_value})",
                )

            return FlowRateSensorData(
                time=reading_time,
                value=reading_value,
            )
        except Exception as exception:
            async_print(
                print_queue=self.print_queue,
                message=f"Error reading flow rate sensor: {exception}",
            )
            return None

    def read_averaged(
        self, number_of_samples: int = 4, should_log: bool = False
    ) -> typing.Optional[FlowRateSensorData]:
        times: list[float] = []
        values: list[float] = []
        max_attempts: int = number_of_samples * 2

        for _ in range(max_attempts):
            if len(values) >= number_of_samples:
                break

            sample: typing.Optional[FlowRateSensorData] = self.read(
                should_log=should_log
            )
            if sample is not None:
                times.append(sample.time)
                values.append(sample.value)

        if len(values) == 0:
            return None

        return FlowRateSensorData(
            time=sum(times) / len(times),
            value=sum(values) / len(values),
        )

    def close(self) -> None:
        if (self.serial_port is not None) and (self.serial_port.is_open):
            if hasattr(self, "sensor"):
                self.sensor.stop_continuous_measurement()

            async_print(print_queue=self.print_queue, message="")
            async_print(
                print_queue=self.print_queue,
                message="Closing Flow Rate Sensor Serial Port...",
            )

            self.serial_port.close()


def printer_worker(print_queue: Queue) -> None:
    erasable_line_count: int = 0

    while True:
        try:
            printable_item = print_queue.get(timeout=0.1)

            if printable_item is None:
                break

            (message, is_erasable, should_erase_previous_now) = printable_item

            if (
                (not is_erasable) or should_erase_previous_now
            ) and erasable_line_count > 0:
                for _ in range(erasable_line_count):
                    sys.stdout.write("\033[F\033[K")
                sys.stdout.flush()
                erasable_line_count = 0

            print(message, flush=True)

            if is_erasable:
                line_count = message.count("\n") + (
                    1 if not message.endswith("\n") else 0
                )
                erasable_line_count += line_count
            else:
                erasable_line_count = 0

        except queue.Empty:
            continue
        except Exception as exception:
            sys.stderr.write(f"Printer worker error: {exception}\n")
            sys.stderr.flush()


def pressure_sensor_worker(
    output_filename_prefix: str,
    pressure_sensor_port_label: str,
    synchronized_trial,
    is_experiment_started_event,
    is_experiment_completed_event,
    synchronized_pressure,
    synchronized_flow_rate,
    print_queue,
) -> None:
    try:
        pressure_sensor: PressureSensor = PressureSensor(
            serial_port_label=pressure_sensor_port_label,
            print_queue=print_queue,
        )

        pressure_sensor_data_filename = f"{output_filename_prefix}-pressure_sensor.csv"

        csv_file_handle = open(pressure_sensor_data_filename, "a", newline="")
        csv_writer = csv.writer(csv_file_handle)
        csv_writer.writerow(
            (
                "trial",
                "time",
                "pressure",
            )
        )

        while not is_experiment_started_event.is_set():
            time.sleep(0.005)
        pressure_sensor.timer.start()

        while not is_experiment_completed_event.is_set():
            # pressure_sensor_data = pressure_sensor.read_averaged(number_of_samples=4)
            pressure_sensor_data = pressure_sensor.read_averaged()
            if pressure_sensor_data is not None:
                with synchronized_pressure.get_lock():
                    synchronized_pressure.value = pressure_sensor_data.value

                with synchronized_trial.get_lock():
                    csv_writer.writerow(
                        (
                            synchronized_trial.value,
                            pressure_sensor_data.time,
                            pressure_sensor_data.value,
                        )
                    )

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="%" * 80)
        async_print(
            print_queue=print_queue,
            message=f"The Pressure Sensor is done with its tasks. Please check the following file for output: {pressure_sensor_data_filename} ...",
        )
        async_print(print_queue=print_queue, message="%" * 80)
    finally:
        try:
            pressure_sensor.close()  # pyright: ignore [reportPossiblyUnboundVariable]
        except NameError:
            pass
        finally:
            if csv_file_handle is not None:
                csv_file_handle.close()


def flow_rate_sensor_worker(
    output_filename_prefix: str,
    flow_rate_sensor_port_label: str,
    synchronized_trial,
    is_experiment_started_event,
    is_experiment_completed_event,
    synchronized_pressure,
    synchronized_flow_rate,
    print_queue,
) -> None:
    try:
        flow_rate_sensor: FlowRateSensor = FlowRateSensor(
            serial_port_label=flow_rate_sensor_port_label,
            print_queue=print_queue,
        )

        flow_rate_sensor_data_filename = (
            f"{output_filename_prefix}-flow_rate_sensor.csv"
        )

        csv_file_handle = open(flow_rate_sensor_data_filename, "a", newline="")
        csv_writer = csv.writer(csv_file_handle)
        csv_writer.writerow(
            (
                "trial",
                "time",
                "flow_rate",
            )
        )

        while not is_experiment_started_event.is_set():
            time.sleep(0.005)
        flow_rate_sensor.timer.start()

        while not is_experiment_completed_event.is_set():
            # flow_rate_sensor_data = flow_rate_sensor.read_averaged(number_of_samples=4)
            flow_rate_sensor_data = flow_rate_sensor.read_averaged()
            if flow_rate_sensor_data is not None:
                with synchronized_flow_rate.get_lock():
                    synchronized_flow_rate.value = flow_rate_sensor_data.value

                with synchronized_trial.get_lock():
                    csv_writer.writerow(
                        (
                            synchronized_trial.value,
                            flow_rate_sensor_data.time,
                            flow_rate_sensor_data.value,
                        )
                    )

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="%" * 80)
        async_print(
            print_queue=print_queue,
            message=f"The Flow Rate Sensor is done with its tasks. Please check the following file for output: {flow_rate_sensor_data_filename} ...",
        )
        async_print(print_queue=print_queue, message="%" * 80)
    finally:
        try:
            flow_rate_sensor.close()  # pyright: ignore [reportPossiblyUnboundVariable]
        except NameError:
            pass
        finally:
            if csv_file_handle is not None:
                csv_file_handle.close()


def pump_worker(
    output_filename_prefix: str,
    grbl_serial_port_label: str,
    waveform_definition: list[tuple[float, float]],
    number_of_trials: int,
    synchronized_trial,
    is_experiment_started_event,
    is_experiment_completed_event,
    synchronized_pressure,
    synchronized_flow_rate,
    print_queue,
) -> None:
    zero_flow_rate_plunger_position: float = (
        0.00  # Declared early for fallback at failure, but assigned later
    )
    try:
        plunger_position_bias: float = 3.0
        pump: Pump = Pump(
            serial_port_label=grbl_serial_port_label,
            print_queue=print_queue,
            z_offset_to_add=plunger_position_bias,
        )
        pump_data_filename = f"{output_filename_prefix}-pump.csv"

        csv_file_handle = open(pump_data_filename, mode="a", newline="")
        csv_writer = csv.writer(csv_file_handle)

        csv_writer.writerow(
            (
                "trial",
                "start_time",
                "end_time",
                "gcode",
            )
        )

        def write_to_csv(
            command_execution_data_list: list[CommandExecutionData],
        ) -> None:
            for command_execution_data in command_execution_data_list:
                with synchronized_trial.get_lock():
                    csv_writer.writerow(
                        (
                            synchronized_trial.value,
                            command_execution_data.start_time,
                            command_execution_data.end_time,
                            command_execution_data.gcode,
                        )
                    )

        def reach_flow_rate(
            target_flow_rate: float,
            sleep_time: float,
            probe_feedrate: float,
            pid_controller: PID,
            last_plunger_position: float = 0.0,
            max_flow_rate_error_fraction: float = 0.02,
            minimum_absolute_flow_rate_error: float = 0.1,
            k_p=0.070,
            k_i=0.030,
            k_d=0.000,
            should_log=True,
        ) -> float:
            with synchronized_flow_rate.get_lock():
                absolute_flow_rate_error: float = abs(
                    target_flow_rate - synchronized_flow_rate.value
                )

            pid_controller.tunings = (k_p, k_i, k_d)
            pid_controller.sample_time = sleep_time
            pid_controller.set_auto_mode(True, last_output=last_plunger_position)

            is_first_pid_log_display: bool = True
            while (absolute_flow_rate_error > minimum_absolute_flow_rate_error) and (
                absolute_flow_rate_error
                > max_flow_rate_error_fraction * abs(target_flow_rate)
            ):
                time.sleep(sleep_time)

                pid_controller.setpoint = target_flow_rate
                with synchronized_flow_rate.get_lock():
                    plunger_position = pid_controller(synchronized_flow_rate.value)
                if plunger_position is None:
                    async_print(print_queue=print_queue, message="!" * 80)
                    async_print(
                        print_queue=print_queue, message="ERROR: PID returned None"
                    )
                    async_print(print_queue=print_queue, message="!" * 80)
                    continue
                plunger_position = numpy.round(plunger_position, 5)

                if should_log:
                    # if not is_first_pid_log_display:
                    #     delete_printed_lines(number_of_lines=9)

                    (p, i, d) = pid_controller.components

                    with synchronized_flow_rate.get_lock():
                        pid_status_lines = [
                            ":" * 80,
                            "PID Controller Status",
                            ":" * 80,
                            f"Target Flow Rate: {target_flow_rate:09.5f} uL/min",
                            f"Current Flow Rate: {synchronized_flow_rate.value:09.5f} uL/min",
                            f"PID components: (P: {p:09.5f} mm, I: {i:09.5f} mm, D: {d:09.5f} mm)",
                            f"Last Plunger Position: {last_plunger_position:09.5f} mm",
                            f"Plunger Position Target: {plunger_position:09.5f} mm",
                            ":" * 80,
                        ]

                    async_print(
                        print_queue=print_queue,
                        message="\n".join(pid_status_lines),
                        is_erasable=True,
                        should_erase_previous_now=True,
                    )

                    is_first_pid_log_display = False

                if abs(plunger_position - last_plunger_position) >= 0.00001:
                    command_execution_data_list: list[CommandExecutionData] = (
                        pump.move_z(
                            position=plunger_position,
                            feedrate=probe_feedrate,
                            should_log=False,
                        )
                    )

                    write_to_csv(
                        command_execution_data_list=command_execution_data_list,
                    )

                last_plunger_position = plunger_position
                with synchronized_flow_rate.get_lock():
                    absolute_flow_rate_error = abs(
                        target_flow_rate - synchronized_flow_rate.value
                    )

            return last_plunger_position

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="@" * 80)
        async_print(print_queue=print_queue, message="Starting the Experiment...")
        async_print(print_queue=print_queue, message="@" * 80)

        is_experiment_started_event.set()
        time.sleep(0.0025)
        pump.timer.start()

        max_flow_rate_error_fraction = 0.02

        pid_controller = PID(
            Kp=0.05,
            Ki=0.05,
            Kd=0.00,
            setpoint=0.0,
            proportional_on_measurement=False,
            output_limits=(
                -2.00 - plunger_position_bias,
                21.00 - plunger_position_bias,
            ),
        )

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="#" * 80)
        async_print(
            print_queue=print_queue,
            message="Determining Plunger Positions from Flow-Rate Targets...",
        )
        async_print(print_queue=print_queue, message="#" * 80)

        number_of_verifications: int = 4
        async_print(print_queue=print_queue, message="Resetting the pump...")
        pump.reset(should_log=False)

        unique_flow_rate_targets: float = []
        for point in waveform_definition:
            if point[1] not in unique_flow_rate_targets:
                unique_flow_rate_targets.append(point[1])
        plunger_position_targets = {}
        plunger_position = 0.0
        plunger_position_sum = 0.0
        zero_padded_unique_flow_rate_targets = [0.0]
        zero_padded_unique_flow_rate_targets.extend(unique_flow_rate_targets)
        zero_padded_unique_flow_rate_targets.sort()
        last_plunger_position: float = 0.0

        for flow_rate_target in zero_padded_unique_flow_rate_targets:
            async_print(print_queue=print_queue, message="")
            async_print(print_queue=print_queue, message="*" * 80)
            async_print(
                print_queue=print_queue,
                message=f"Aiming for the flow rate target: {flow_rate_target} uL/min",
            )
            async_print(print_queue=print_queue, message="*" * 80)

            plunger_position_sum = 0.0

            for verified_times, probe_feedrate, sleep_time in zip(
                range(0, number_of_verifications),
                [640.0, 160.0, 40.0, 10.0],
                [2.40, 1.20, 0.60, 0.30],
            ):
                async_print(
                    print_queue=print_queue,
                    message="Waiting 20s for the flow to stabilize...",
                )
                command_execution_data_list = []
                # command_execution_data_list = pump.pause(duration=20, should_log=False)
                # write_to_csv(
                #     command_execution_data_list=command_execution_data_list,
                # )
                async_print(print_queue=print_queue, message="Done waiting...")

                plunger_position = reach_flow_rate(
                    target_flow_rate=flow_rate_target,
                    sleep_time=sleep_time,
                    probe_feedrate=probe_feedrate,
                    pid_controller=pid_controller,
                    last_plunger_position=last_plunger_position,
                    max_flow_rate_error_fraction=max_flow_rate_error_fraction,
                    should_log=True,
                )

                if (number_of_verifications - verified_times) <= 3.0:
                    plunger_position_sum += (
                        plunger_position  # pyright: ignore[reportOperatorIssue]
                    )

                last_plunger_position = plunger_position

                async_print(print_queue=print_queue, message="-" * 80)
                async_print(
                    print_queue=print_queue,
                    message=f"Reached the target flow rate at Plunger Position: {plunger_position} mm",
                )
                async_print(print_queue=print_queue, message="-" * 80)

            plunger_position = numpy.round(plunger_position_sum / 3.0, 5)
            plunger_position_targets[flow_rate_target] = plunger_position
            async_print(print_queue=print_queue, message="")
            async_print(print_queue=print_queue, message="=" * 80)
            async_print(
                print_queue=print_queue,
                message=f"Mean of last three Plunger Positions: {plunger_position} mm",
            )
            async_print(print_queue=print_queue, message="=" * 80)

        plunger_position_waveform_definition: list[tuple[float, float]] = []
        for point in waveform_definition:
            plunger_position_waveform_definition.append(
                (
                    point[0],
                    plunger_position_targets[point[1]],
                )
            )

        async_print(print_queue=print_queue, message="")
        async_print(
            print_queue=print_queue,
            message=f"Plunger Position Targets for various Flow Rate Targets: \n {
            pandas.DataFrame(
                list(plunger_position_targets.items()),
                columns= ["Flow Rate Target [uL/min]", "Plunger Position Target [mm]"]
            )
        }",
        )

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="#" * 80)
        async_print(
            print_queue=print_queue,
            message="Determining Feedrates from Plunger Displacements...",
        )
        async_print(print_queue=print_queue, message="#" * 80)

        def get_feedrate_from_trapezoidal_planning(
            displacement: float,  # mm
            duration: float,  # min
            maximum_acceleration: float = 1800000,  # mm/min^2
            maximum_feedrate: float = 2500.0,  # mm/min
            minimum_feedrate: float = 1e-3,  # mm/min
            dead_time: float = 0.00029350591788065246,  # min
            should_print_warnings: bool = True,
        ) -> tuple[float, bool]:
            is_clamped: bool = False
            distance: float = abs(displacement)  # mm
            total_time: float = duration - dead_time  # min
            minimum_total_time: float = 2.0 * numpy.sqrt(
                distance / maximum_acceleration
            )
            if total_time < minimum_total_time:
                if should_print_warnings:
                    async_print(
                        print_queue=print_queue,
                        message=f"Total time not enough to reach the destination for displacement: {displacement}, average_speed: {average_speed}",
                    )
                total_time = minimum_total_time + 1e-9

            maximum_acceleration_times_total_time: float = (
                maximum_acceleration * total_time
            )
            trapezoidal_peak_speed: float = 0.5 * (
                maximum_acceleration_times_total_time
                - numpy.sqrt(
                    maximum_acceleration_times_total_time
                    * maximum_acceleration_times_total_time
                    - 4.0 * maximum_acceleration * distance
                )
            )

            feedrate: float = trapezoidal_peak_speed
            if feedrate < minimum_feedrate:
                feedrate = minimum_feedrate
                is_clamped = True
                if should_print_warnings:
                    async_print(print_queue=print_queue, message="")
                    async_print(print_queue=print_queue, message="!" * 80)
                    async_print(
                        print_queue=print_queue,
                        message=f"Clamped to Min Feedrate for displacement: {displacement}, duration: {duration}",
                    )
                    async_print(print_queue=print_queue, message="!" * 80)
            elif feedrate > maximum_feedrate:
                feedrate = maximum_feedrate
                is_clamped = True
                if should_print_warnings:
                    async_print(print_queue=print_queue, message="")
                    async_print(print_queue=print_queue, message="!" * 80)
                    async_print(
                        print_queue=print_queue,
                        message=f"Clamped to Max Feedrate for displacement: {displacement}, duration: {duration}",
                    )
                    async_print(print_queue=print_queue, message="!" * 80)
            return (feedrate, is_clamped)

        plunger_displacement_waveform_definition: list[tuple[float, float]] = []
        for current_point, next_point in pairwise(waveform_definition):
            plunger_displacement_waveform_definition.append(
                (
                    next_point[0] - current_point[0],
                    next_point[1] - current_point[1],
                )
            )
        print(plunger_displacement_waveform_definition)

        feedrates = {}
        for plunger_displacement_task in plunger_displacement_waveform_definition:
            feedrates[plunger_displacement_task] = numpy.round(
                get_feedrate_from_trapezoidal_planning(
                    duration=plunger_displacement_task[0],
                    displacement=abs(plunger_displacement_task[1]),
                )[0],
                3,
            )

        async_print(print_queue=print_queue, message="")
        feedrate_dataframe: pandas.DataFrame = pandas.DataFrame(
            [
                [duration, plunger_displacement, feedrate]
                for ((duration, plunger_displacement), feedrate) in feedrates.items()
            ],
            columns=[
                "Duration [min]",
                "Plunger Displacement [uL/min]",
                "Feed Rate [mm/min]",
            ],
        )
        async_print(
            print_queue=print_queue,
            message=f"Feedrates for various Plunger Displacements: \n {feedrate_dataframe}",
        )

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="#" * 80)
        async_print(
            print_queue=print_queue,
            message="Executing Flow Rate Waveform...",
        )
        async_print(print_queue=print_queue, message="#" * 80)

        command_execution_data_list: list[CommandExecutionData] = []

        # Reminder to self to not reset the pump here! It traps air at zero position and messes up both the PID controller and the flow rate
        # async_print(print_queue=print_queue, message= "Resetting the pump...")
        # pump.reset()

        async_print(
            print_queue=print_queue, message="Waiting 10s for the flow to stabilize..."
        )
        _ = pump.pause(duration=10)
        async_print(print_queue=print_queue, message="Done waiting...")

        baseline_plunger_position = plunger_displacement_waveform_definition[0][1]

        pid_controller.set_auto_mode(False)
        command_execution_data_list += pump.move_z(
            position=baseline_plunger_position,
            feedrate=160.0,
        )
        write_to_csv(command_execution_data_list=command_execution_data_list)
        pid_controller.set_auto_mode(True, last_output=baseline_plunger_position)

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="-" * 80)
        async_print(print_queue=print_queue, message="Pausing for 10s")
        async_print(print_queue=print_queue, message="-" * 80)
        _ = pump.pause(duration=10)
        async_print(print_queue=print_queue, message="Done pausing...")
        async_print(print_queue=print_queue, message="-" * 80)

        baseline_plunger_position = reach_flow_rate(
            target_flow_rate=waveform_definition[0][1],
            sleep_time=1.20,
            probe_feedrate=160.0,
            pid_controller=pid_controller,
            last_plunger_position=baseline_plunger_position,
            max_flow_rate_error_fraction=0.02,
        )

        for trial in range(1, number_of_trials + 1):

            with synchronized_trial.get_lock():
                synchronized_trial.value = trial

            async_print(print_queue=print_queue, message="")
            async_print(print_queue=print_queue, message="-" * 80)
            async_print(print_queue=print_queue, message=f"Beginning trial: {trial}")
            async_print(print_queue=print_queue, message="-" * 80)

            command_execution_data_list = []

            for point, plunger_displacement_task in zip(
                plunger_position_waveform_definition[1:],
                plunger_displacement_waveform_definition,
            ):

                if plunger_displacement_task[1] != 0:
                    command_execution_data_list += pump.move_z(
                        position=point[1],
                        feedrate=feedrates[plunger_displacement_task],
                    )
                else:
                    command_execution_data_list += pump.pause(
                        duration=plunger_displacement_task[0],
                    )

                write_to_csv(
                    command_execution_data_list=command_execution_data_list,
                )

            with synchronized_trial.get_lock():
                synchronized_trial.value = 0

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="-" * 80)
        async_print(print_queue=print_queue, message="Pausing for 10s")
        async_print(print_queue=print_queue, message="-" * 80)
        _ = pump.pause(duration=10)
        async_print(print_queue=print_queue, message="Done pausing...")
        async_print(print_queue=print_queue, message="-" * 80)

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="%" * 80)
        async_print(
            print_queue=print_queue,
            message=f"The Pump is done with its tasks. Please check the following file for output: {pump_data_filename} ...",
        )
        async_print(print_queue=print_queue, message="%" * 80)
    finally:
        try:
            pump.move_z(  # pyright: ignore [reportPossiblyUnboundVariable]
                position=0.0, feedrate=1000.0, should_log=False
            )
            pump.close()  # pyright: ignore [reportPossiblyUnboundVariable]
        except NameError:
            pass
        finally:
            is_experiment_completed_event.set()
            if csv_file_handle is not None:
                csv_file_handle.close()


def main() -> None:
    # =========================================================================
    # Waveform is defined below
    # =========================================================================

    pressure_sensor_serial_port_label = "COM6"
    flow_rate_sensor_serial_port_label = "COM4"
    grbl_serial_port_label = "COM7"

    waveform_definition: list[tuple[float, float]] = [
        (0.00, 5.00),
        (0.10, 15.00),
        (0.30, 0.00),
        (0.50, 8.00),
        (0.60, 5.00),
        (1.10, 5.00),
    ]

    number_of_trials = 6

    # =========================================================================

    timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
    output_filename_prefix = f"experimental_data-multiphasic_waveform-{timestamp}"

    print_queue = Queue(maxsize=1000)

    synchronized_trial = Value(
        "i",
        0,
    )
    is_experiment_started_event = Event()
    is_experiment_completed_event = Event()
    synchronized_pressure = Value(
        "d",
        0.0,
    )
    synchronized_flow_rate = Value(
        "d",
        0.0,
    )

    printer_process: Process = Process(target=printer_worker, args=(print_queue,))

    pressure_sensor_process: Process = Process(
        target=pressure_sensor_worker,
        args=(
            output_filename_prefix,
            pressure_sensor_serial_port_label,
            synchronized_trial,
            is_experiment_started_event,
            is_experiment_completed_event,
            synchronized_pressure,
            synchronized_flow_rate,
            print_queue,
        ),
    )

    flow_rate_sensor_process: Process = Process(
        target=flow_rate_sensor_worker,
        args=(
            output_filename_prefix,
            flow_rate_sensor_serial_port_label,
            synchronized_trial,
            is_experiment_started_event,
            is_experiment_completed_event,
            synchronized_pressure,
            synchronized_flow_rate,
            print_queue,
        ),
    )

    pump_process: Process = Process(
        target=pump_worker,
        args=(
            output_filename_prefix,
            grbl_serial_port_label,
            waveform_definition,
            number_of_trials,
            synchronized_trial,
            is_experiment_started_event,
            is_experiment_completed_event,
            synchronized_pressure,
            synchronized_flow_rate,
            print_queue,
        ),
    )

    try:
        printer_process.start()
        pressure_sensor_process.start()
        flow_rate_sensor_process.start()
        pump_process.start()
    finally:
        pump_process.join()
        pressure_sensor_process.join()
        flow_rate_sensor_process.join()

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="@" * 80)
        async_print(print_queue=print_queue, message="Experiment done!")
        async_print(print_queue=print_queue, message="@" * 80)

        async_print(print_queue=print_queue, message="")
        async_print(print_queue=print_queue, message="Exiting the program...")
        async_print(print_queue=print_queue, message="")

        time.sleep(0.5)
        print_queue.put(None)
        printer_process.join()


if __name__ == "__main__":
    main()